In [1]:
import pandas as pd
import numpy as np
import os

# 1. CARGA DEL DATASET REAL (El "Molde")
df_real = pd.read_csv('../data/processed/dataset_entrenamiento_tawos.csv')

print(f"Dataset molde cargado: {df_real.shape[0]} tareas.")
print("Analizando distribuciones estadísticas reales para la simulación...")

# Extraemos las probabilidades reales de los tipos de tarea para que el simulador sea realista
prob_tipos = df_real['Issue_Type'].value_counts(normalize=True).to_dict()
tipos_tarea = list(prob_tipos.keys())
probabilidades = list(prob_tipos.values())

Dataset molde cargado: 142151 tareas.
Analizando distribuciones estadísticas reales para la simulación...


In [2]:
# --- BANCO DE TÍTULOS SINTÉTICOS (contexto semántico para el LLM) ---
TITULOS_POR_TIPO = {
    'Bug': [
        "Error de validación en formulario de alta de usuario",
        "Fallo en cálculo de totales en módulo de facturación",
        "NullPointerException al cargar el dashboard principal",
        "Timeout en llamada al endpoint de autenticación",
        "Datos duplicados al sincronizar con el servicio externo",
        "Excepción no controlada al exportar informe a PDF",
        "Sesión no invalidada correctamente al cerrar sesión",
    ],
    'Enhancement': [
        "Mejorar rendimiento de la consulta de histórico de pedidos",
        "Añadir paginación al listado de clientes",
        "Optimizar carga inicial del dashboard reduciendo llamadas API",
        "Reducir tiempo de respuesta del buscador global",
    ],
    'Task': [
        "Migrar configuración de entorno a variables de entorno",
        "Actualizar dependencias de seguridad del backend",
        "Refactorizar módulo de logging para cumplir estándar corporativo",
        "Configurar pipeline de CI/CD en el nuevo entorno de staging",
    ],
    'Story': [
        "Como usuario quiero filtrar el listado por fecha de creación",
        "Implementar exportación a CSV en el panel de informes",
        "Como admin quiero gestionar roles y permisos desde la UI",
        "Añadir notificaciones push al módulo de alertas",
    ],
    'Other': [
        "Revisión de documentación técnica del API REST",
        "Sesión de refinamiento de backlog Q3",
        "Investigación de alternativas para el módulo de reportes",
    ],
    'Epic': [
        "Rediseño del módulo de autenticación y gestión de sesiones",
        "Integración con sistema de pagos externo (Stripe)",
        "Migración de arquitectura monolítica a microservicios",
    ]
}

# 2. EL MOTOR DE GENERACIÓN SINTÉTICA
def generar_sprint_sintetico(nombre_proyecto, sprint_id, num_tareas, nivel_caos='normal'):
    """
    Genera un Sprint ficticio parametrizado para probar la API y los Dashboards.
    nivel_caos: 'perfecto' (0% riesgo), 'normal' (realidad), 'catastrofico' (muchos bloqueos)
    """
    sprint_data = []
    
    if nivel_caos == 'perfecto':
        prob_cambio = 0.01
        max_bloqueos = 0
    elif nivel_caos == 'catastrofico':
        prob_cambio = 0.60
        max_bloqueos = 5
    else:
        prob_cambio = 0.15
        max_bloqueos = 2

    for i in range(1, num_tareas + 1):
        # 1. Datos básicos
        issue_key = f"{nombre_proyecto[:3].upper()}-{sprint_id}0{i}"
        issue_type = np.random.choice(tipos_tarea, p=probabilidades)
        titulo = np.random.choice(TITULOS_POR_TIPO.get(issue_type, TITULOS_POR_TIPO['Other']))
        
        # 2. Estimaciones (forzamos esfuerzo alto en catastrofico para superar el percentil 75)
        if nivel_caos == 'catastrofico':
            story_points = np.random.choice([8, 13])
            esfuerzo_base = np.random.randint(29500, 50000)  # ← supera el p75 real (29.380 min)
        else:
            story_points = np.random.choice([1, 2, 3, 5, 8, 13])
            esfuerzo_base = story_points * np.random.randint(240, 480)
        
        # 3. Simulación de Riesgos
        title_changed = np.random.binomial(1, prob_cambio)
        desc_changed = np.random.binomial(1, prob_cambio)
        sp_changed = np.random.binomial(1, prob_cambio / 2)
        bloqueos = np.random.randint(0, max_bloqueos + 1)
        
        # 4. Simulación del tiempo real invertido
        penalizacion_riesgo = 1.0 + (bloqueos * 0.3) + (desc_changed * 0.5)
        
        if np.random.rand() > 0.05:
            tiempo_resolucion = int(esfuerzo_base * np.random.uniform(1.1, 2.5) * penalizacion_riesgo)
        else:
            tiempo_resolucion = int(esfuerzo_base * np.random.uniform(0.7, 1.0))
            
        # 5. Tiempos intermedios
        in_progress = int(tiempo_resolucion * np.random.uniform(0.3, 0.9))

        sprint_data.append({
            'Issue_Key': issue_key,
            'Title': titulo,
            'Issue_Type': issue_type,
            'Project_ID': 999,
            'Project_Name': nombre_proyecto,
            'Sprint_ID': sprint_id,
            'Sprint_State': 'ACTIVE',
            'Story_Point': float(story_points),
            'Total_Effort_Minutes': float(esfuerzo_base),
            'In_Progress_Minutes': float(in_progress),
            'Resolution_Time_Minutes': float(tiempo_resolucion),
            'Title_Changed_After_Estimation': title_changed,
            'Description_Changed_After_Estimation': desc_changed,
            'Story_Point_Changed_After_Estimation': sp_changed,
            'Blocker_Count': bloqueos
        })
        
    return pd.DataFrame(sprint_data)

print("✅ Motor de generación sintética (Sprint Simulator) inicializado.")

✅ Motor de generación sintética (Sprint Simulator) inicializado.


In [3]:
# 3. CREACIÓN DE ESCENARIOS DE PRUEBA

# Directorio de salida
out_dir = '../data/synthetic/'
os.makedirs(out_dir, exist_ok=True)

# Escenario A: Un Sprint normal de 30 tareas
df_sprint_normal = generar_sprint_sintetico("Proyecto Alpha", 1, 30, nivel_caos='normal')
df_sprint_normal.to_csv(f"{out_dir}sprint_normal.csv", index=False)

# Escenario B: El "Sprint Catastrófico" (Para probar las alertas rojas de la PMO)
df_sprint_caos = generar_sprint_sintetico("Proyecto Omega", 2, 45, nivel_caos='catastrofico')
df_sprint_caos.to_csv(f"{out_dir}sprint_catastrofico.csv", index=False)

# Escenario C: El "Sprint Perfecto" (Para engordar la clase minoritaria y que la IA aprenda)
df_sprint_perfecto = generar_sprint_sintetico("Proyecto Zen", 3, 50, nivel_caos='perfecto')
df_sprint_perfecto.to_csv(f"{out_dir}sprint_perfecto.csv", index=False)

print(f"✅ ¡Archivos generados con éxito en la carpeta '{out_dir}'!")
print(f" - Sprint Normal: {df_sprint_normal.shape[0]} tareas.")
print(f" - Sprint Catastrófico: {df_sprint_caos.shape[0]} tareas.")
print(f" - Sprint Perfecto: {df_sprint_perfecto.shape[0]} tareas.")

✅ ¡Archivos generados con éxito en la carpeta '../data/synthetic/'!
 - Sprint Normal: 30 tareas.
 - Sprint Catastrófico: 45 tareas.
 - Sprint Perfecto: 50 tareas.


In [4]:
# 4. CREACIÓN DE UN PROYECTO COMPLETO (Para la vista de la PMO)

sprints_proyecto = []

# Simulamos la evolución del "Proyecto Delta" a lo largo de 4 Sprints
# Sprint 1: Arranque normal
sprints_proyecto.append(generar_sprint_sintetico("Proyecto Delta", 1, 30, nivel_caos='normal'))

# Sprint 2: Empieza a torcerse un poco
sprints_proyecto.append(generar_sprint_sintetico("Proyecto Delta", 2, 35, nivel_caos='normal'))

# Sprint 3: La crisis del proyecto (ideal para que salten las alarmas de la PMO)
sprints_proyecto.append(generar_sprint_sintetico("Proyecto Delta", 3, 40, nivel_caos='catastrofico'))

# Sprint 4: Se aplican medidas y se estabiliza
sprints_proyecto.append(generar_sprint_sintetico("Proyecto Delta", 4, 25, nivel_caos='perfecto'))

# Unimos todos los sprints en un único archivo maestro del proyecto
df_proyecto_completo = pd.concat(sprints_proyecto, ignore_index=True)
df_proyecto_completo.to_csv(f"{out_dir}proyecto_completo.csv", index=False)

print(
    f"✅ ¡Proyecto completo generado! "
    f"'proyecto_completo.csv' con {df_proyecto_completo.shape[0]} "
    f"tareas en total distribuidas en 4 Sprints."
)

✅ ¡Proyecto completo generado! 'proyecto_completo.csv' con 130 tareas en total distribuidas en 4 Sprints.
